# Kronos Alpha Terminal — research notebook

Explore forecasts, regimes and signals interactively. Runs in **mock mode** by default
(no model files needed). Set `KAT_MOCK_MODE=auto` and install `requirements-kronos.txt`
plus the vendored Kronos repo to use the real model.

> Mock output is for plumbing only — never claim edge from it.

In [ ]:
import os, sys
os.environ.setdefault('KAT_MOCK_MODE', 'true')
sys.path.insert(0, os.path.abspath('../backend'))
from app.config import get_config_store
from app.db.session import new_session
from app.repositories import candles_repo
from app.kronos.adapter import KronosAdapter
from app.strategy.regime_detector import detect_regime
from app.strategy.signal_scoring import score_signal
store = get_config_store()

In [ ]:
# Load candles for one symbol (ingest first via `make ingest_crypto`).
db = new_session()
symbol, tf = 'BTC/USDT', '1h'
rows = candles_repo.get_candles(db, symbol, tf, ascending=True, limit=400)
df = candles_repo.candles_to_df(rows)
print(len(df), 'candles')
df.tail()

In [ ]:
adapter = KronosAdapter(store.kronos())
dist = adapter.forecast(df, symbol=symbol, timeframe=tf)
print('mode:', dist.mode.value, '| p_up:', round(dist.p_up, 3), '| median_ret:', round(dist.median_return, 4))
import matplotlib.pyplot as plt
plt.plot(dist.median_path, label='median')
plt.fill_between(range(len(dist.median_path)), dist.q10_path, dist.q90_path, alpha=0.2, label='q10-q90')
plt.axhline(dist.last_close, ls='--', c='gray'); plt.legend(); plt.title(f'{symbol} forecast ({dist.mode.value})'); plt.show()

In [ ]:
regime = detect_regime(df, symbol=symbol, timeframe=tf)
sig = score_signal(forecast=dist, regime=regime, df=df, strategy=store.strategy(), risk=store.risk())
print('regime:', regime.regime.value, '-', regime.explanation)
print('signal:', sig.status.value, '| edge:', sig.edge_score, '| reasons:', sig.reason_codes)
db.close()